In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

In [2]:
DATA = Path("E:/data analysis/End-to-End Supply Chain Inventory Optimization System/data/supply_chain_inventory_125k.csv")
df = pd.read_csv(DATA, parse_dates=["transaction_date"])

In [3]:
df.head()

,transaction_id,transaction_date,sku_id,warehouse_id,supplier_id,category,region,quantity_sold,unit_cost,unit_price,...,eoq,ordering_cost,holding_cost_per_unit_year,stockout_flag,overstock_flag,replenishment_flag,order_status,forecast_demand,forecast_error_pct,fulfillment_lead_time_days
0,76730,2024-01-01,210,6,3,Home Appliances,North,14,267.37,416.26,...,121,93.52,48.50,0,0,0,Completed,10.70,23.57,10
1,14275,2024-01-01,162,5,24,Home Appliances,Central,8,146.30,176.52,...,150,83.81,30.77,1,0,1,Completed,11.63,45.38,5
2,20231,2024-01-01,345,1,19,Personal Care,North,5,309.95,359.55,...,79,93.32,72.69,0,0,0,Completed,6.87,37.40,5
3,115042,2024-01-01,93,7,31,Home Appliances,South,27,311.51,449.88,...,108,49.13,74.49,1,0,1,Cancelled,22.76,15.70,15
4,64451,2024-01-01,251,1,24,Personal Care,North,28,185.90,299.09,...,177,77.33,53.38,1,0,1,Completed,30.37,8.46,14


-----Check Data Quality---


In [4]:
print("Shape:", df.shape)
print("\nMissing values:\n", df.isna().sum())
print("\nDuplicate transaction ID's:", df["transaction_id"].duplicated().sum())

Shape: (125000, 30)

Missing values:
 transaction_id                0
transaction_date              0
sku_id                        0
warehouse_id                  0
supplier_id                   0
category                      0
region                        0
quantity_sold                 0
unit_cost                     0
unit_price                    0
sales_value                   0
inventory_on_hand             0
inventory_value               0
lead_time_days                0
lead_time_std_days            0
supplier_on_time_pct          0
demand_mean_daily             0
demand_std_daily              0
safety_stock                  0
reorder_point                 0
eoq                           0
ordering_cost                 0
holding_cost_per_unit_year    0
stockout_flag                 0
overstock_flag                0
replenishment_flag            0
order_status                  0
forecast_demand               0
forecast_error_pct            0
fulfillment_lead_time_days    0
dt

--------KPI Calculation----------


In [5]:
total_revenue = df["sales_value"].sum()
inventory_value = df["inventory_value"].sum()
units_sold = df["quantity_sold"].sum()
stockout_rate = df["stockout_flag"].mean() * 100
overstock_rate = df["overstock_flag"].mean() * 100
avg_lead_time = df["fulfillment_lead_time_days"].mean()

print(f"\nRevenue: ${total_revenue:,.2f}")
print(f"Inventory value: ${inventory_value:,.2f}")
print(f"Units sold: {units_sold:,}")
print(f"Stockout transaction rate: {stockout_rate:.2f}%")
print(f"Overstock transaction rate: {overstock_rate:.2f}%")
print(f"Average fulfillment lead time: {avg_lead_time:.2f} days")



Revenue: $586,317,939.86
Inventory value: $4,124,503,177.81
Units sold: 2,251,217
Stockout transaction rate: 50.70%
Overstock transaction rate: 8.73%
Average fulfillment lead time: 8.51 days


------Inventory Turnover-------

In [6]:
cogs = (df["quantity_sold"] * df["unit_cost"]).sum()
avg_inventory = df["inventory_value"].mean()
inventory_turnover = cogs / avg_inventory
print(f"Inventory turnover: {inventory_turnover:.2f}x")

Inventory turnover: 12668.72x


-------Replenishment Simulation-------

In [7]:
df["baseline_stockout"] = (
    df["inventory_on_hand"] <= df["demand_mean_daily"] * df["lead_time_days"]
).astype(int)

df["optimized_stockout"] = (
    df["inventory_on_hand"] <= df["reorder_point"]
).astype(int)

baseline_rate = df["baseline_stockout"].mean()
optimized_rate = df["optimized_stockout"].mean()

print(f"Baseline stockout rate: {baseline_rate:.2%}")
print(f"Policy stockout trigger rate: {optimized_rate:.2%}")


Baseline stockout rate: 36.84%
Policy stockout trigger rate: 50.94%


In [8]:
sim = df.copy()
sim["projected_inventory_after_policy"] = np.where(
    sim["inventory_on_hand"] <= sim["reorder_point"],
    sim["inventory_on_hand"] + sim["eoq"],
    sim["inventory_on_hand"]
)
sim["projected_stockout"] = (
    sim["projected_inventory_after_policy"] < sim["demand_mean_daily"]
).astype(int)

risk_reduction = (
    1 - sim["projected_stockout"].mean() / max(sim["baseline_stockout"].mean(), 1e-9)
) * 100

print(f"Projected stockout-risk reduction: {risk_reduction:.2f}%")


Projected stockout-risk reduction: 100.00%


------Warehouse Summary-------

In [9]:
warehouse = (
    df.groupby(["warehouse_id", "region"], as_index=False)
      .agg(
          revenue=("sales_value", "sum"),
          inventory_value=("inventory_value", "sum"),
          units_sold=("quantity_sold", "sum"),
          avg_lead_time=("fulfillment_lead_time_days", "mean"),
          stockout_events=("stockout_flag", "sum"),
          overstock_events=("overstock_flag", "sum"),
      )
)

warehouse["inventory_turnover_proxy"] = (
    warehouse["units_sold"] / warehouse["inventory_value"]
)



--------SKU Alerts----------

In [10]:
alerts = (
    df.groupby(["sku_id", "warehouse_id", "category"], as_index=False)
      .agg(
          current_inventory=("inventory_on_hand", "mean"),
          reorder_point=("reorder_point", "mean"),
          safety_stock=("safety_stock", "mean"),
          recommended_order_qty=("eoq", "mean"),
          inventory_value=("inventory_value", "sum"),
      )
)

alerts["status"] = np.select(
    [
        alerts["current_inventory"] <= alerts["safety_stock"],
        alerts["current_inventory"] <= alerts["reorder_point"],
        alerts["current_inventory"] > alerts["reorder_point"] + 2 * alerts["recommended_order_qty"],
    ],
    ["CRITICAL", "REORDER", "OVERSTOCK"],
    default="HEALTHY"
)

OUT = Path("../data/processed")
OUT.mkdir(exist_ok=True)

warehouse.to_csv(OUT / "warehouse_summary.csv", index=False)
alerts.to_csv(OUT / "replenishment_alerts.csv", index=False)

print("\nExported Power BI-ready tables:")
print(" - warehouse_summary.csv")
print(" - replenishment_alerts.csv")


Exported Power BI-ready tables:
 - warehouse_summary.csv
 - replenishment_alerts.csv


In [11]:
warehouse

,warehouse_id,region,revenue,inventory_value,units_sold,avg_lead_time,stockout_events,overstock_events,inventory_turnover_proxy
0,1,North,59643011.77,4.178322e+08,228594,8.492762,6357,1093,0.000547
1,2,South,58987332.97,4.182382e+08,226695,8.512955,6298,1143,0.000542
2,3,East,57643326.53,4.036728e+08,222303,8.491160,6273,1034,0.000551
3,4,West,57470355.66,4.100633e+08,221498,8.555547,6239,1065,0.000540
4,5,Central,59846804.11,4.194144e+08,228490,8.559965,6481,1078,0.000545
5,6,North,58618860.86,4.093393e+08,225114,8.556196,6405,1126,0.000550
6,7,South,59284559.99,4.101835e+08,226610,8.433822,6374,1067,0.000552
7,8,East,57289698.28,4.067329e+08,221685,8.490187,6301,1092,0.000545
8,9,West,57735419.80,4.098085e+08,221393,8.494328,6236,1085,0.000540
9,10,Central,59798569.89,4.192183e+08,228835,8.507829,6405,1130,0.000546


In [12]:
alerts

,sku_id,warehouse_id,category,current_inventory,reorder_point,safety_stock,recommended_order_qty,inventory_value,status
0,1,1,Personal Care,159.520000,173.240000,49.920000,163.520000,381093.28,REORDER
1,1,2,Personal Care,168.736842,204.526316,61.578947,176.000000,306365.36,REORDER
2,1,3,Personal Care,212.111111,197.333333,55.666667,181.277778,364848.08,HEALTHY
3,1,4,Personal Care,194.423077,187.153846,48.461538,172.230769,483055.80,HEALTHY
4,1,5,Personal Care,189.500000,174.642857,47.000000,162.571429,507041.36,HEALTHY
...,...,...,...,...,...,...,...,...,...
4995,500,6,Personal Care,168.862069,172.172414,47.275862,183.655172,375550.93,REORDER
4996,500,7,Personal Care,156.826087,194.869565,51.086957,199.260870,276620.83,REORDER
4997,500,8,Personal Care,159.370370,175.037037,49.555556,195.703704,329997.07,REORDER
4998,500,9,Personal Care,170.827586,192.724138,55.344828,205.172414,379922.26,REORDER
